# Unfallatlas 2025: Data Preparation and Splits

This notebook turns the raw accident data into **train / validation / test splits** that the modelling notebooks start from.
Findings behind the choices made here are in `tiziano_eda.ipynb`; column meanings are in `data/DSB_Unfallatlas_EN.pdf`.

**Prediction task:** given the circumstances of an accident with personal injury, predict whether someone was
**killed or seriously injured** (`target = UKATEGORIE <= 2`, about 15.7 % positives).
The original 3-class `UKATEGORIE` is kept in every split, so a multi-class model can use the same files.

Workflow:
1. Load and clean (stateless steps only, so they are safe before the split)
2. Create the target and stateless features
3. Split 70 / 15 / 15, stratified by `UKATEGORIE`
4. Save the splits to `data/splits/`

Anything that is *learned* from data (scaling, encoding, target encoding, resampling) belongs **after** the split and is fitted on the training set only.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42
SPLIT_DIR = Path("data/splits")

## 1. Load and clean

* `UIDENTSTLAE` is an ID with leading zeros, so it is read as a string.
* Coordinates use a German decimal comma.

In [2]:
raw = pd.read_csv("data/Unfallorte_2025.csv", sep=";", decimal=",", dtype={"UIDENTSTLAE": str})
print(raw.shape)
raw.head()

(273007, 25)


,UIDENTSTLAE,ULAND,UREGBEZ,UKREIS,UGEMEINDE,UJAHR,UMONAT,USTUNDE,UWOCHENTAG,UKATEGORIE,...,IstPKW,IstFuss,IstKrad,IstGkfz,IstSonstige,LINREFX,LINREFY,XGCSWGS84,YGCSWGS84,PLST
0,01250509171013352025,1,0,60,7,2025,5,15,6,3,...,1,0,0,0,0,578658.6459,5.975442e+06,10.197759,53.921403,1
1,01250405161013752025,1,0,62,53,2025,4,14,7,3,...,0,0,0,0,0,577747.0359,5.933251e+06,10.173275,53.542399,1
2,01250118171013142025,1,0,56,15,2025,1,9,7,3,...,1,0,0,0,0,543500.2814,5.955820e+06,9.659670,53.749197,1
3,01250326152013912025,1,0,55,28,2025,3,16,4,3,...,1,1,0,0,0,601362.0656,6.003469e+06,10.552708,54.169272,1
4,01250225145013362025,1,0,58,116,2025,2,17,3,3,...,0,0,0,0,0,565151.5561,6.035316e+06,10.005090,54.461326,1


### Merge the date fields into `datetime`

There is no day-of-month column. The day is encoded in the ID (`LL YYMMDD...`), which the EDA confirmed against
the weekday column (>99.99 % agreement). Year, month and hour come from their own columns.

In [3]:
df = raw.copy()
df["datetime"] = pd.to_datetime(
    dict(
        year=df["UJAHR"],
        month=df["UMONAT"],
        day=df["UIDENTSTLAE"].str[6:8].astype(int),
        hour=df["USTUNDE"],
    )
)

# Unfallatlas weekday: 1 = Sunday ... 7 = Saturday; pandas: 0 = Monday ... 6 = Sunday
weekday_check = (df["datetime"].dt.dayofweek + 1) % 7 + 1
print("rows where derived date disagrees with UWOCHENTAG:", (weekday_check != df["UWOCHENTAG"]).sum())

rows where derived date disagrees with UWOCHENTAG: 8


The handful of inconsistent rows is negligible and kept. The original date columns are dropped,
because everything is now derived from `datetime`.

In [4]:
df = df.drop(columns=["UJAHR", "UMONAT", "USTUNDE", "UWOCHENTAG"])

### Drop columns that are not accident characteristics

| column | reason |
| --- | --- |
| `LINREFX`, `LINREFY` | same location as the WGS84 coordinates, in another projection |
| `PLST` | geocoding quality flag from data processing; `PLST = 2` occurs only for bicycle accidents |

`UIDENTSTLAE` stays in the splits as an identifier, but it is **not a feature** (it encodes the date and region).

In [5]:
df = df.drop(columns=["LINREFX", "LINREFY", "PLST"])
df = df.rename(columns={"XGCSWGS84": "lon", "YGCSWGS84": "lat"})

## 2. Target and stateless features

These features are computed row by row and learn nothing from other rows, so they are safe before the split.

* **Time:** hour, month, weekday (0 = Monday), weekend flag, plus sine/cosine encodings so that 23 h sits next to 0 h and December next to January.
* **Region key:** `UKREIS` and `UGEMEINDE` are only unique within their state, so they are combined into
  official-code-style keys (`kreis_key` = state + admin region + district, `gemeinde_key` = plus municipality). They are high-cardinality, so use them with target encoding or a frequency encoding (fitted on train only).
* **Involvement:** number of road-user groups involved.

In [6]:
df["target"] = (df["UKATEGORIE"] <= 2).astype(int)

t = df["datetime"]
df["hour"] = t.dt.hour
df["month"] = t.dt.month
df["weekday"] = t.dt.dayofweek
df["is_weekend"] = (df["weekday"] >= 5).astype(int)
df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)
df["month_sin"] = np.sin(2 * np.pi * (df["month"] - 1) / 12)
df["month_cos"] = np.cos(2 * np.pi * (df["month"] - 1) / 12)

df["kreis_key"] = (
    df["ULAND"].astype(str).str.zfill(2)
    + df["UREGBEZ"].astype(str)
    + df["UKREIS"].astype(str).str.zfill(2)
)
df["gemeinde_key"] = df["kreis_key"] + df["UGEMEINDE"].astype(str).str.zfill(3)

INVOLVED = ["IstRad", "IstPKW", "IstFuss", "IstKrad", "IstGkfz", "IstSonstige"]
df["n_involved"] = df[INVOLVED].sum(axis=1)

print("districts:", df["kreis_key"].nunique(), "| municipalities:", df["gemeinde_key"].nunique())
df.head()

districts: 513 | municipalities: 9479


,UIDENTSTLAE,ULAND,UREGBEZ,UKREIS,UGEMEINDE,UKATEGORIE,UART,UTYP1,ULICHTVERH,IstStrassenzustand,...,month,weekday,is_weekend,hour_sin,hour_cos,month_sin,month_cos,kreis_key,gemeinde_key,n_involved
0,01250509171013352025,1,0,60,7,3,5,3,0,0,...,5,4,0,-0.707107,-0.707107,0.866025,-5.000000e-01,01060,01060007,1
1,01250405161013752025,1,0,62,53,3,0,7,0,0,...,4,5,1,-0.500000,-0.866025,1.000000,6.123234e-17,01062,01062053,1
2,01250118171013142025,1,0,56,15,3,5,3,0,1,...,1,5,1,0.707107,-0.707107,0.000000,1.000000e+00,01056,01056015,1
3,01250326152013912025,1,0,55,28,3,6,2,0,0,...,3,2,0,-0.866025,-0.500000,0.866025,5.000000e-01,01055,01055028,2
4,01250225145013362025,1,0,58,116,3,2,6,0,0,...,2,1,0,-0.965926,-0.258819,0.500000,8.660254e-01,01058,01058116,1


### Feature groups

These lists define what the modelling notebooks feed to a `ColumnTransformer`.
`UART`, `UTYP1` and `ULAND` are nominal codes: their numbers have no order, so one-hot encode them for linear
models (tree models can use them as they are). `ULICHTVERH` and `IstStrassenzustand` are ordered (daylight → darkness, dry → winter-slippery).

In [7]:
TARGET = "target"
TARGET_MULTICLASS = "UKATEGORIE"
ID_COLS = ["UIDENTSTLAE", "datetime"]

NOMINAL = ["ULAND", "UART", "UTYP1"]
ORDINAL = ["ULICHTVERH", "IstStrassenzustand"]
BINARY = INVOLVED + ["is_weekend"]
NUMERIC = ["hour_sin", "hour_cos", "month_sin", "month_cos", "n_involved", "lon", "lat"]
HIGH_CARDINALITY = ["kreis_key", "gemeinde_key"]
FEATURES = NOMINAL + ORDINAL + BINARY + NUMERIC + HIGH_CARDINALITY

# raw codes and plain time columns are kept for error analysis, but are not in FEATURES
AUXILIARY = ["UREGBEZ", "UKREIS", "UGEMEINDE", "hour", "month", "weekday"]

df = df[ID_COLS + FEATURES + AUXILIARY + [TARGET_MULTICLASS, TARGET]]
assert df.columns.is_unique and df.notna().all().all()
print(len(FEATURES), "features")

21 features


## 3. Train / validation / test split

* **70 % train**: fit models and cross-validate for hyperparameter tuning.
* **15 % validation**: compare models and do error analysis without touching the test set.
* **15 % test**: used **once**, for the final score of the chosen model.

The split is random and stratified by `UKATEGORIE`. With only 0.9 % fatal accidents, stratification keeps the
same share of each severity level in every split. A random split is fine here because accidents are separate events
and the goal is to explain severity, not to forecast the future. An alternative is a time-based split (e.g. train on January–October and test on November–December), which would test whether the model holds up over the seasons.

In [8]:
train, temp = train_test_split(
    df, test_size=0.30, stratify=df[TARGET_MULTICLASS], random_state=RANDOM_STATE
)
val, test = train_test_split(
    temp, test_size=0.50, stratify=temp[TARGET_MULTICLASS], random_state=RANDOM_STATE
)

splits = {"train": train, "val": val, "test": test}
summary = pd.DataFrame({
    name: {
        "rows": len(s),
        "share_%": round(len(s) / len(df) * 100, 1),
        "% killed": round((s[TARGET_MULTICLASS] == 1).mean() * 100, 2),
        "% seriously injured": round((s[TARGET_MULTICLASS] == 2).mean() * 100, 2),
        "% target = 1": round(s[TARGET].mean() * 100, 2),
    }
    for name, s in splits.items()
}).T
summary

,rows,share_%,% killed,% seriously injured,% target = 1
train,191104.0,70.0,0.91,14.81,15.72
val,40951.0,15.0,0.91,14.81,15.72
test,40952.0,15.0,0.91,14.81,15.72


In [9]:
# sanity checks: no row in two splits, nothing lost
ids = {name: set(s["UIDENTSTLAE"]) for name, s in splits.items()}
assert not (ids["train"] & ids["val"]) and not (ids["train"] & ids["test"]) and not (ids["val"] & ids["test"])
assert sum(len(s) for s in splits.values()) == len(df)
print("splits are disjoint and complete")

splits are disjoint and complete


## 4. Save the splits

Saved as gzip-compressed CSV (pandas reads them directly). The IDs stay strings, the date is parsed back to `datetime`.

In [10]:
SPLIT_DIR.mkdir(parents=True, exist_ok=True)
for name, s in splits.items():
    s.to_csv(SPLIT_DIR / f"{name}.csv.gz", index=False)
    print(f"saved {SPLIT_DIR / f'{name}.csv.gz'}: {len(s):,} rows")

saved data/splits/train.csv.gz: 191,104 rows


saved data/splits/val.csv.gz: 40,951 rows


saved data/splits/test.csv.gz: 40,952 rows


## How to load the splits in a modelling notebook

Copy this cell. The feature lists above are repeated so the modelling notebook is self-contained.

In [11]:
def load_split(name):
    return pd.read_csv(
        f"data/splits/{name}.csv.gz",
        dtype={"UIDENTSTLAE": str, "kreis_key": str, "gemeinde_key": str},
        parse_dates=["datetime"],
    )


train = load_split("train")
val = load_split("val")

X_train, y_train = train[FEATURES], train[TARGET]
X_val, y_val = val[FEATURES], val[TARGET]
print(X_train.shape, X_val.shape, f"positives in train: {y_train.mean():.1%}")

(191104, 21) (40951, 21) positives in train: 15.7%


## Notes for modelling

* **Metric:** classes are imbalanced (≈ 16 % positives), so accuracy is misleading (always predicting "slight" scores 84 %).
  Use PR-AUC / average precision, recall and F1 for the positive class, and pick a decision threshold on the validation set.
* **Imbalance:** try `class_weight="balanced"` first; under- or oversampling only inside the training folds.
* **Simple baselines to beat:** a `DummyClassifier`, and a rule such as "motorcycle involved or vehicle left the carriageway"
  (both stood out as high-severity in the EDA).
* **Leakage check:** all features describe the accident itself and are known once it is recorded. Nothing describes the outcome.